# Structural audit — cặp Qwen3-Embedding-0.6B → MiniLMv2-L6-H384 (analysis)

Notebook này **không train gì**. Nó đọc một run của `audit_runs_qwen_minilm.ipynb`
(probe dump từng epoch của mỗi arm, teacher trên probe set, target map đã fit, điểm
test cuối) và dựng mọi bảng/figure của `docs/experiments_and_figures.md` mà dữ liệu
hiện có cho phép. Đổi một metric ở đây không bao giờ kéo theo train lại.

| Cell | Claim | Sản phẩm |
|---|---|---|
| 4 | Props 1–2 | bảng target variants (retained energy, angular distortion, STS ceiling) · **Figure 6** |
| 5 | C1 | **Table 2** (interface ablation) · **Figure 2** (dissociation) · **Figure 7** (distortion → score) |
| 6 | C2 | **Table 4** (ladder) · **Figure 3** (ladder chuẩn hoá) · **Figure 11** (k-NN vs N) · **Figure 8** (pairwise cosine) · **Figure 9** (residual spectrum) · **Figure 14** (anisotropy) · **Figure 10** (2-D, minh hoạ) |
| 7 | C3 | **Figure 4** (depth heatmap) · **Figure 12** (TwoNN, cosine theo layer) · **Table 5** |
| 8 | G | **Table 6** (null band, PR) · **Figure 5** (gauge interpolation — có điểm nào vẽ điểm đó) |
| 9 | M | **Figure 13** (truncation) · corpus sweep · **Table 7** (matched-TALAS) |

Quy ước: mọi metric so sánh hai không gian đều bất biến với phép xoay trực giao
trừ rung 1 (cosine tới target) — rung 1 là nơi duy nhất gauge lộ ra. Arm nhiều seed
được gộp mean ± std theo `base`. Arm chưa chạy (thiếu code) tự vắng mặt khỏi figure;
caption nào cần arm đó thì in cảnh báo. Mọi output ghi vào `<run>/analysis/`.


In [ ]:
# 1. Cấu hình phân tích.
from pathlib import Path

# Tên run của audit_runs_qwen_minilm.ipynb. None = run audit_* mới nhất trong OUTPUT_BASE.
RUN_NAME = None
SAVE_TO_GOOGLE_DRIVE = False
# Thư mục chứa các run (None = <repo>/runs, hoặc Drive khi SAVE_TO_GOOGLE_DRIVE).
OUTPUT_BASE_OVERRIDE = None

# Ladder (§C2).
KS = (1, 10, 50)                      # k của k-NN overlap
PROBE_SIZES = (1000, 4000, 16000, "full")  # N của Figure 11
LADDER_N = 4000                       # N dùng cho Table 4 / Figure 3 (kNN); rung 2/4 tự subsample
GRAM_ROWS = 4096                      # số hàng cho Gram / CKA / Procrustes
H0_ROWS, H0_DRAWS = 2000, 3           # H0 barcode: subsample x draws
LADDER_RUNG_ORDER = ("cos_to_target", "gram_corr", "linear_cka", "procrustes", "knn@10", "mutual_knn@10", "h0_w1")
SEED = 0

# Target variants (Figure 6/7) và truncation (Figure 13).
K_SWEEP = (64, 128, 256, 384)
TRUNCATION_DIMS = (32, 64, 96, 128, 192, 256, 320, 384)
N_PAIRS = 200_000                     # số cặp ngẫu nhiên cho Figure 8/14 và anisotropy
# Arm dùng cho các figure nhiều panel (theo `base`; arm không tồn tại tự bị bỏ qua).
PANEL_BASES = ("pca__procrustes", "pca__none", "random__none__d0", "mrl_prefix__none",
               "learned_t2s__lr1", "learned_s2t__lr1", "pca__mse", "simcse_only")
DEPTH_BASES = ("pca__procrustes", "ours+anchor_k", "ours+lasd", "freeze_lower", "learned_t2s__lr1", "simcse_only")
COLOUR_TASK = "emotion_test"          # subset có nhãn để tô màu Figure 10


In [ ]:
# 2. Repo + thư viện (umap-learn là tuỳ chọn cho Figure 10; thiếu thì dùng PCA 2-D).
import subprocess
import sys

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
cwd = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in (cwd, cwd.parent) if (p / "main.py").is_file() and (p / "distiller.py").is_file()), None
)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
try:
    import umap  # noqa: F401
    HAVE_UMAP = True
except Exception:
    HAVE_UMAP = False
print(f"Project directory: {PROJECT_DIR}; umap: {HAVE_UMAP}")


In [ ]:
# 3. Nạp run: config, probe set, teacher/student-init trên probe, cache train, arm.
import json
import math
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display

from src.cache_teacher import cache_filename, load_cached_embeddings
from src.probe_set import eval_pairs_in_probe, probe_digest
from src import structural_audit as sa

plt.style.use("seaborn-v0_8-whitegrid")
try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if OUTPUT_BASE_OVERRIDE:
    OUTPUT_BASE = Path(OUTPUT_BASE_OVERRIDE)
elif IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

if RUN_NAME is None:
    runs = sorted(p for p in OUTPUT_BASE.glob("audit_*") if (p / "run_config.json").is_file())
    assert runs, f"Không có run audit_* nào trong {OUTPUT_BASE}"
    RUN_ROOT = runs[-1]
else:
    RUN_ROOT = OUTPUT_BASE / RUN_NAME
CONFIG = json.loads((RUN_ROOT / "run_config.json").read_text(encoding="utf-8"))
ARMS_ROOT = Path(CONFIG["arms_root"])
ANALYSIS_DIR = RUN_ROOT / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)
D_S = None
print(f"Run: {RUN_ROOT.name}\n  {CONFIG['teacher_model']} -> {CONFIG['student_model']} on {CONFIG['dataset_tag']}")

PROBE = pd.read_csv(CONFIG["probe_path"], keep_default_na=False)
assert probe_digest(PROBE) == CONFIG["probe_digest"], "probe set trên đĩa không khớp run_config"
N_PROBE = len(PROBE)
teacher_payload = torch.load(CONFIG["probe_teacher_path"], map_location="cpu", weights_only=False)
assert teacher_payload["probe_digest"] == CONFIG["probe_digest"]
TEACHER = teacher_payload["embeddings"].float()          # [N, d_T] chưa normalize
TEACHER_UNIT = F.normalize(TEACHER, dim=-1)
STUDENT_INIT = torch.load(CONFIG["student_init_path"], map_location="cpu", weights_only=False)
CORE_INDEX = STUDENT_INIT["core_index"].numpy()
D_T = TEACHER.shape[1]
D_S = STUDENT_INIT["final"].shape[1]
print(f"Probe: {N_PROBE} câu (core {len(CORE_INDEX)}); d_T={D_T}, d_S={D_S}")

# Cache teacher của corpus train (để fit PCA-k / random-k / MRL-k đúng như training).
cache_path = Path(CONFIG["cache_dir"]) / cache_filename(
    teacher_model_name=CONFIG["teacher_model"], pooling_method=CONFIG["teacher_pooling"],
    train_data_path=CONFIG["train_data"], max_length=int(CONFIG["matched_hp"]["max_length"]), normalize=True,
)
if cache_path.is_file():
    CACHE, _ = load_cached_embeddings(str(cache_path))
    CACHE = CACHE.float()
else:
    CACHE = None
    warnings.warn(f"Không thấy cache teacher {cache_path}; các variant PCA-k/random-k sẽ fit trên teacher của probe set")

# Arm: plan + trạng thái + summary.
ARMS = {}
for arm in CONFIG["arms"]:
    arm_dir = ARMS_ROOT / arm["name"]
    summary_path = arm_dir / "train_summary.json"
    entry = dict(arm, dir=arm_dir, done=summary_path.is_file())
    entry["summary"] = json.loads(summary_path.read_text(encoding="utf-8")) if entry["done"] else None
    ARMS[arm["name"]] = entry
DONE = [name for name, a in ARMS.items() if a["done"]]
BASES = list(dict.fromkeys(ARMS[n]["base"] for n in DONE))
SEEDS_OF = {base: [n for n in DONE if ARMS[n]["base"] == base] for base in BASES}
print(f"{len(DONE)}/{len(ARMS)} arm có probe dump: {BASES}")
missing = [n for n, a in ARMS.items() if not a["done"]]
if missing:
    print("Chưa có:", ", ".join(f"{n} ({'thiếu code' if ARMS[n]['needs'] else 'chưa chạy'})" for n in missing))


def score_of(name, key="avg_all"):
    test = (ARMS[name]["summary"] or {}).get("final_test_summary") or {}
    return None if test.get(key) is None else 100.0 * float(test[key])


def final_train(name, key):
    train = (ARMS[name]["summary"] or {}).get("final_train") or {}
    return train.get(key)


_final_cache = {}


def load_final(name, epoch=None):
    """Embedding layer cuối trên cả probe set (float32) ở epoch (None = cuối)."""
    if name not in _final_cache:
        _final_cache[name] = torch.load(ARMS[name]["dir"] / "probe_final.pt", map_location="cpu", weights_only=False)
    payload = _final_cache[name]
    assert payload["probe_digest"] == CONFIG["probe_digest"]
    epochs = sorted(payload["final"])
    return payload["final"][epochs[-1] if epoch is None else epoch].float(), epochs


def load_layers(name):
    payload = torch.load(ARMS[name]["dir"] / "probe_layers.pt", map_location="cpu", weights_only=False)
    return {epoch: layers.float() for epoch, layers in payload["layers"].items()}


def learned_map(name, epoch=None):
    path = ARMS[name]["dir"] / "learned_map.pt"
    if not path.is_file():
        return None
    payload = torch.load(path, map_location="cpu", weights_only=False)
    epochs = payload["epochs"]
    return payload["weight"][epochs[-1] if epoch is None else epoch]


def arm_target(name):
    """Target của arm trên probe set: (tau [N, d_S] normalised, kind) hoặc (None, kind).

    fixed        : map đã lưu (P, mean, R) áp lên teacher của probe set
    learned_t2s  : W cuối áp lên teacher (target *học được*)
    learned_s2t  : không có target trong không gian student (so sánh trong không gian teacher)
    none         : simcse/talas — không có target
    """
    projection_path = ARMS[name]["dir"] / "teacher_projection.pt"
    if projection_path.is_file():
        saved = sa.load_saved_projection(projection_path)
        if saved.get("projection") is not None:
            return sa.targets_from_saved(TEACHER, saved), "fixed"
        if saved.get("projection_type") == "learned_t2s":
            W = learned_map(name)
            return (F.normalize(TEACHER @ W.T, dim=-1) if W is not None else None), "learned_t2s"
        return None, "learned_s2t"
    return None, "none"


OURS = next((b for b in ("pca__procrustes",) if b in SEEDS_OF), None)
OURS_ARM = SEEDS_OF[OURS][0] if OURS else None
TARGET_OURS = arm_target(OURS_ARM)[0] if OURS_ARM else None
if TARGET_OURS is None:
    warnings.warn("Không có arm pca__procrustes: ceiling của ladder và target tham chiếu sẽ dùng PCA fit lại trên cache")
    P, mean = sa.fit_variant(CACHE if CACHE is not None else TEACHER, "pca", D_S)
    TARGET_OURS = sa.apply_map(TEACHER, P, mean=mean)

# Cặp câu của các task STS / pair trên probe set (điểm tính thẳng từ embedding).
PAIR_TASKS = {stem: f"data/test_set/{stem}.csv" for stem in ("sick_test", "sts12_test", "stsb_test", "mrpc_test", "scitail_test", "wic_test")}
PAIRS = eval_pairs_in_probe(PROJECT_DIR, PROBE, PAIR_TASKS)
STS_TASKS = [t for t, s in PAIRS.items() if s["kind"] == "sts"]
PAIR_CLS_TASKS = [t for t, s in PAIRS.items() if s["kind"] == "pair"]
for task, spec in PAIRS.items():
    if spec["missing"]:
        print(f"[warn] {task}: {spec['missing']} cặp không nằm trong probe set")


def mean_task_scores(emb, tasks):
    scores = sa.pair_task_scores(emb, {t: PAIRS[t] for t in tasks})
    return float(np.nanmean(list(scores.values()))), scores


def by_base(frame, value_columns):
    """mean ± std theo base (seed band); giữ thứ tự xuất hiện."""
    grouped = frame.groupby("base", sort=False)[list(value_columns)].agg(["mean", "std", "count"])
    grouped.columns = [f"{c}_{s}" for c, s in grouped.columns]
    return grouped.reset_index()


def savefig(fig, name):
    path = ANALYSIS_DIR / name
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"  -> {path}")


RNG = np.random.default_rng(SEED)
SUBSETS = {}
for size in PROBE_SIZES:
    n = N_PROBE if size == "full" else min(int(size), N_PROBE)
    SUBSETS[size] = np.sort(RNG.choice(N_PROBE, size=n, replace=False))
LADDER_IDX = SUBSETS[LADDER_N] if LADDER_N in SUBSETS else np.sort(RNG.choice(N_PROBE, size=min(LADDER_N, N_PROBE), replace=False))


In [ ]:
# 4. Target variants: retained energy, angular distortion, STS ceiling — Figure 6.
#
# Hai loại target: (a) map mà mỗi arm map-đóng-băng đã thực sự train theo (đọc từ
# teacher_projection.pt), và (b) họ PCA-k / random-k / MRL-k fit trên cache train ở
# nhiều rank k. Distortion = RMS chênh lệch cosine từng cặp giữa target và teacher đầy
# đủ (Gram distance); ceiling = điểm STS của chính target, không có student.
FIT_SOURCE = CACHE if CACHE is not None else TEACHER
target_rows = []
TARGETS = {}   # key -> tau [N, d] normalised


def describe_target(key, tau, kind, k, energy, note=""):
    TARGETS[key] = tau
    sts_avg, sts = mean_task_scores(tau, STS_TASKS)
    target_rows.append({
        "target": key, "kind": kind, "k": k, "retained_energy": energy,
        "distortion": sa.gram_rmse(tau[CORE_INDEX], TEACHER[CORE_INDEX], max_rows=GRAM_ROWS, seed=SEED),
        "sts_avg": 100 * sts_avg, **{f"sts_{t}": 100 * v for t, v in sts.items()}, "note": note,
    })


describe_target("teacher", TEACHER_UNIT, "teacher", D_T, 1.0, "trần: teacher đầy đủ")
for base in BASES:
    name = SEEDS_OF[base][0]
    tau, kind = arm_target(name)
    if tau is None or kind != "fixed":
        continue
    projection = (ARMS[name]["summary"] or {}).get("projection") or {}
    describe_target(base, tau, projection.get("projection_type", "fixed"), D_S, projection.get("explained_energy"), "map của arm")
for kind in ("pca", "random", "mrl_prefix"):
    for k in K_SWEEP:
        P, mean = sa.fit_variant(FIT_SOURCE, kind, k, seed=0)
        describe_target(f"{kind}_k{k}", sa.apply_map(TEACHER, P, mean=mean), kind, k,
                        sa.retained_energy(FIT_SOURCE, P), "fit trên cache train")
# learned_t2s: distortion của target *đã fit* (W áp lên teacher) — nằm ngoài đường cong (Figure 7).
for base in BASES:
    name = SEEDS_OF[base][0]
    tau, kind = arm_target(name)
    if kind == "learned_t2s" and tau is not None:
        describe_target(base, tau, "learned_t2s", D_S, None, "W cuối áp lên teacher")

targets_table = pd.DataFrame(target_rows)
targets_table.to_csv(ANALYSIS_DIR / "targets.csv", index=False)
display(targets_table.style.format(precision=4, na_rep="-"))

# Figure 6: phổ teacher (trái) và energy-vs-distortion của ba họ (phải).
sigma = sa.singular_values(FIT_SOURCE[: min(len(FIT_SOURCE), 50000)], center=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(np.arange(1, len(sigma) + 1), sigma / sigma[0])
axes[0].axvline(D_S, color="k", ls="--", lw=1, label=f"d_S = {D_S}")
axes[0].set(title="Teacher singular values (centred)", xlabel="index", ylabel="sigma_i / sigma_1")
axes[0].legend(frameon=False)
for kind, marker in (("pca", "o"), ("random", "s"), ("mrl_prefix", "^")):
    sub = targets_table[(targets_table["kind"] == kind) & (targets_table["note"] == "fit trên cache train")]
    axes[1].plot(sub["distortion"], sub["retained_energy"], marker=marker, label=kind)
    for _, r in sub.iterrows():
        axes[1].annotate(f"k={int(r['k'])}", (r["distortion"], r["retained_energy"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[1].set(title="What each interface keeps", xlabel="angular distortion (Gram RMSE vs teacher)", ylabel="retained energy")
axes[1].legend(frameon=False)
fig.tight_layout(); savefig(fig, "fig06_teacher_spectrum.png"); plt.show()


In [ ]:
# 5. C1 — Table 2 (interface ablation), Figure 2 (dissociation), Figure 7 (distortion -> score).
rows = []
for name in DONE:
    arm = ARMS[name]
    final, _ = load_final(name)
    tau, kind = arm_target(name)
    projection = (arm["summary"] or {}).get("projection") or {}
    row = {
        "arm": name, "base": arm["base"], "group": arm["group"], "seed": arm["seed"], "kind": kind,
        "retained_energy": projection.get("explained_energy"),
        "distortion": (targets_table.set_index("target")["distortion"].get(arm["base"]) if kind in ("fixed", "learned_t2s") else np.nan),
        "final_loss_end": final_train(name, "loss_end"),
        "final_cos": final_train(name, "cos_final"),
        "effective_rank": sa.effective_rank(final[CORE_INDEX]),
        "anisotropy": sa.anisotropy(final[CORE_INDEX], n_pairs=N_PAIRS // 4, seed=SEED),
        "avg_all": score_of(name), "avg_iod": score_of(name, "avg_iod"), "avg_ood": score_of(name, "avg_ood"),
    }
    rows.append(row)
arm_table = pd.DataFrame(rows)
arm_table.to_csv(ANALYSIS_DIR / "arm_metrics.csv", index=False)
table2 = by_base(arm_table, ["retained_energy", "distortion", "final_loss_end", "effective_rank", "avg_all"])
table2.insert(1, "group", [ARMS[SEEDS_OF[b][0]]["group"] for b in table2["base"]])
table2.to_csv(ANALYSIS_DIR / "table2_interface.csv", index=False)
print("TABLE 2 — interface ablation (mean ± std over seeds)")
display(table2.style.format(precision=4, na_rep="-"))

# Figure 2: x = endpoint loss cuối, y = điểm, size = effective rank. Kỳ vọng: map cố
# định ở góc trên-phải, learned_t2s dưới-trái với marker nhỏ nhất.
c1 = table2[table2["final_loss_end_mean"].notna() & table2["avg_all_mean"].notna()]
fig, ax = plt.subplots(figsize=(7.5, 5))
size = 20 + 300 * (c1["effective_rank_mean"] / max(c1["effective_rank_mean"].max(), 1e-9))
ax.scatter(c1["final_loss_end_mean"], c1["avg_all_mean"], s=size, alpha=0.75, edgecolor="k")
ax.errorbar(c1["final_loss_end_mean"], c1["avg_all_mean"], xerr=c1["final_loss_end_std"].fillna(0), yerr=c1["avg_all_std"].fillna(0), fmt="none", ecolor="gray", lw=0.8)
for _, r in c1.iterrows():
    ax.annotate(r["base"], (r["final_loss_end_mean"], r["avg_all_mean"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set(title="Figure 2 — dissociation: endpoint loss vs downstream (marker = effective rank)", xlabel="final endpoint loss (train)", ylabel="avg_all (test, x100)")
fig.tight_layout(); savefig(fig, "fig02_dissociation.png"); plt.show()

# Figure 7: distortion của target -> điểm. Map cố định phải đơn điệu; learned_t2s lệch khỏi đường.
c7 = table2[table2["distortion_mean"].notna() & table2["avg_all_mean"].notna()].copy()
c7["kind"] = [arm_table[arm_table["base"] == b]["kind"].iloc[0] for b in c7["base"]]
fig, ax = plt.subplots(figsize=(7, 4.5))
for kind, marker in (("fixed", "o"), ("learned_t2s", "X")):
    sub = c7[c7["kind"] == kind]
    ax.errorbar(sub["distortion_mean"], sub["avg_all_mean"], yerr=sub["avg_all_std"].fillna(0), fmt=marker, capsize=3, label=kind)
    for _, r in sub.iterrows():
        ax.annotate(r["base"], (r["distortion_mean"], r["avg_all_mean"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set(title="Figure 7 — target distortion predicts transfer", xlabel="angular distortion of target (Gram RMSE vs teacher)", ylabel="avg_all (x100)")
ax.legend(frameon=False)
fig.tight_layout(); savefig(fig, "fig07_distortion_vs_score.png"); plt.show()
if not any(b.startswith("pca_k") for b in BASES):
    print("[note] Chưa có arm pca_k{64,128,256} (thiếu code --projection_rank): Figure 7 chỉ có các map full-rank.")


In [ ]:
# 6. C2 — ladder: Table 4, Figure 3, Figure 11, Figure 8, Figure 9, Figure 14, Figure 10.
#
# Rung 1 dùng target của chính arm (map đóng băng: P,R đã lưu; learned_t2s: W cuối;
# learned_s2t: cosine trong không gian teacher sau khi nâng student bằng W; simcse/
# talas: target của ours làm tham chiếu — ghi rõ ở cột `rung1_ref`). Rung 2–4 so
# student với teacher ĐẦY ĐỦ, không cần target. Ceiling = PCA target vs teacher;
# floor = simcse_only (nếu thiếu: student chưa train).
def ladder_of(student_full, target_full, idx, cos_override=None):
    values = sa.ladder(student_full[idx], TEACHER[idx], target=None if target_full is None else target_full[idx],
                       ks=KS, gram_rows=GRAM_ROWS, h0_rows=H0_ROWS, h0_draws=H0_DRAWS, seed=SEED)
    if cos_override is not None:
        values["cos_to_target"] = cos_override
    return values


ceiling = ladder_of(TARGET_OURS, TARGET_OURS, LADDER_IDX)
floor_name = SEEDS_OF["simcse_only"][0] if "simcse_only" in SEEDS_OF else None
floor_emb = load_final(floor_name)[0] if floor_name else STUDENT_INIT["final"].float()
floor = ladder_of(floor_emb, TARGET_OURS, LADDER_IDX)
print("floor =", floor_name or "student init (không có simcse_only)")

ladder_rows = []
for name in DONE:
    final, _ = load_final(name)
    tau, kind = arm_target(name)
    rung1_ref, cos_override = "own", None
    if kind == "learned_s2t":
        W = learned_map(name)
        cos_override = float(sa.cosine_to_target(final[LADDER_IDX] @ W.T, TEACHER[LADDER_IDX]).mean()) if W is not None else np.nan
        rung1_ref = "teacher space (lifted)"
    elif tau is None:
        tau, rung1_ref = TARGET_OURS, "ours' target"
    values = ladder_of(final, tau, LADDER_IDX, cos_override)
    ladder_rows.append({"arm": name, "base": ARMS[name]["base"], "group": ARMS[name]["group"], "seed": ARMS[name]["seed"],
                        "kind": kind, "rung1_ref": rung1_ref, "avg_all": score_of(name), **values})
ladder_table = pd.DataFrame(ladder_rows)
ladder_table.to_csv(ANALYSIS_DIR / "table4_ladder_by_arm.csv", index=False)
rung_cols = [c for c in ladder_table.columns if c in sa.RUNG_SIGN or c.startswith("knn@") or c.startswith("mutual_knn@")]
table4 = by_base(ladder_table, rung_cols + ["avg_all"])
table4.to_csv(ANALYSIS_DIR / "table4_ladder.csv", index=False)
print("TABLE 4 — ladder values (mean ± std over seeds); ceiling / floor:")
display(pd.DataFrame([{"which": "ceiling (PCA target)", **ceiling}, {"which": f"floor ({floor_name or 'init'})", **floor}]).style.format(precision=4))
display(table4.style.format(precision=4, na_rep="-"))

# Figure 3: ladder chuẩn hoá (ceiling = 1, floor = 0), một đường mỗi base.
fig, ax = plt.subplots(figsize=(9, 4.8))
for _, r in table4.iterrows():
    ys = [sa.normalise_rung(r[f"{rung}_mean"], ceiling[rung], floor[rung]) for rung in LADDER_RUNG_ORDER]
    ax.plot(range(len(LADDER_RUNG_ORDER)), ys, marker="o", label=r["base"], lw=1.5 if r["base"] == OURS else 1)
ax.axhline(1, color="k", lw=0.8, ls="--"); ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xticks(range(len(LADDER_RUNG_ORDER))); ax.set_xticklabels(LADDER_RUNG_ORDER, rotation=20)
ax.set(title="Figure 3 — structural ladder (PCA target = 1, floor = 0)", ylabel="teacher agreement (normalised)")
ax.legend(ncol=4, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22))
fig.tight_layout(); savefig(fig, "fig03_ladder.png"); plt.show()
for missing in ("ours+gram", "ours+lasd", "ours+anchor_k"):
    if missing not in SEEDS_OF:
        print(f"[note] {missing} chưa có (thiếu code) — Figure 3 chưa có arm relational/layer-wise.")

# Figure 11: k-NN overlap vs probe size N (mutual-kNN không phụ thuộc quy mô thì thứ tự phải giữ).
K_SWEEP_FIG11 = 10
teacher_tables = {size: sa.knn_indices(TEACHER[idx], K_SWEEP_FIG11) for size, idx in SUBSETS.items()}
fig11_rows = []
for base in BASES:
    for name in SEEDS_OF[base]:
        final, _ = load_final(name)
        for size, idx in SUBSETS.items():
            fig11_rows.append({"base": base, "arm": name, "N": len(idx), "size": str(size),
                               "knn": sa.knn_overlap(final[idx], TEACHER[idx], K_SWEEP_FIG11, neighbours_b=teacher_tables[size])})
for size, idx in SUBSETS.items():
    fig11_rows.append({"base": "PCA target (ceiling)", "arm": "-", "N": len(idx), "size": str(size),
                       "knn": sa.knn_overlap(TARGET_OURS[idx], TEACHER[idx], K_SWEEP_FIG11, neighbours_b=teacher_tables[size])})
fig11 = pd.DataFrame(fig11_rows)
fig11.to_csv(ANALYSIS_DIR / "fig11_knn_vs_probe_size.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4.5))
for base, sub in fig11.groupby("base", sort=False):
    agg = sub.groupby("N")["knn"].agg(["mean", "std"]).reset_index()
    ax.errorbar(agg["N"], agg["mean"], yerr=agg["std"].fillna(0), marker="o", capsize=3, label=base, ls="--" if "ceiling" in base else "-")
ax.set_xscale("log"); ax.set(title=f"Figure 11 — k-NN overlap (k={K_SWEEP_FIG11}) vs probe size", xlabel="N", ylabel="k-NN overlap with teacher")
ax.legend(fontsize=8, frameon=False, ncol=2)
fig.tight_layout(); savefig(fig, "fig11_knn_vs_N.png"); plt.show()

# Figure 8: 2-D density (teacher cosine, student cosine) trên cặp ngẫu nhiên của core.
pairs = sa.random_pairs(len(CORE_INDEX), N_PAIRS, seed=SEED)
teacher_cos = sa.pairwise_cosines(TEACHER[CORE_INDEX], pairs)
panels = [b for b in PANEL_BASES if b in SEEDS_OF]
fig, axes = plt.subplots(1, max(len(panels), 1), figsize=(3.2 * max(len(panels), 1), 3.4), squeeze=False)
for ax, base in zip(axes[0], panels):
    student_cos = sa.pairwise_cosines(load_final(SEEDS_OF[base][0])[0][CORE_INDEX], pairs)
    ax.hist2d(teacher_cos, student_cos, bins=80, cmap="magma", cmin=1)
    ax.plot([-1, 1], [-1, 1], color="w", lw=0.6, ls="--")
    ax.set(title=base, xlabel="teacher cosine", ylabel="student cosine", xlim=(-0.5, 1), ylim=(-0.5, 1))
fig.suptitle("Figure 8 — pairwise-cosine agreement"); fig.tight_layout(); savefig(fig, "fig08_pairwise_cosine.png"); plt.show()

# Figure 9: phổ residual Z - T so với phổ của T (Observation 3.1).
fig, ax = plt.subplots(figsize=(7, 4.5))
for base in panels:
    name = SEEDS_OF[base][0]
    tau, kind = arm_target(name)
    if tau is None:
        continue
    spectrum = sa.residual_spectrum(load_final(name)[0][CORE_INDEX], tau[CORE_INDEX])
    ax.semilogy(spectrum["residual"] / spectrum["target"][0], label=f"{base} residual")
target_sigma = sa.singular_values(TARGET_OURS[CORE_INDEX], center=False)
ax.semilogy(target_sigma / target_sigma[0], color="k", ls="--", label="PCA target spectrum")
ax.set(title="Figure 9 — residual spectrum (relative to target sigma_1)", xlabel="index", ylabel="sigma_i / sigma_1(target)")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout(); savefig(fig, "fig09_residual_spectrum.png"); plt.show()

# Figure 14: histogram cosine cặp ngẫu nhiên (anisotropy).
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(teacher_cos, bins=120, histtype="step", density=True, label="teacher", lw=1.5)
ax.hist(sa.pairwise_cosines(TARGET_OURS[CORE_INDEX], pairs), bins=120, histtype="step", density=True, label="PCA target", lw=1.5)
for base in panels:
    ax.hist(sa.pairwise_cosines(load_final(SEEDS_OF[base][0])[0][CORE_INDEX], pairs), bins=120, histtype="step", density=True, label=base)
ax.set(title="Figure 14 — distribution of pairwise cosines", xlabel="cosine", ylabel="density"); ax.legend(fontsize=8, frameon=False)
fig.tight_layout(); savefig(fig, "fig14_anisotropy.png"); plt.show()

# Figure 10 (minh hoạ, không phải bằng chứng). Teacher (d_T) và student (d_S) sống ở hai
# không gian khác nhau, nên "fit một lần rồi transform" chỉ có nghĩa trong một không
# gian chung: ảnh PCA·R của teacher (target của ours). Reducer fit trên target đó; mỗi
# panel student được Procrustes-align về toạ độ target trước khi transform (mọi metric
# của audit đều bất biến với phép xoay này, nên nó chỉ bỏ gauge chứ không bỏ gì khác).
from src.teacher_projection import fit_gauge_alignment

labelled = PROBE.index[(PROBE["task"] == COLOUR_TASK) & PROBE["core"].astype(bool)].to_numpy()
if len(labelled) < 50:
    labelled = CORE_INDEX[:2000]
labels = PROBE.loc[labelled, "label"].astype(str).to_numpy()
palette = {lab: i for i, lab in enumerate(sorted(set(labels)))}
colours = np.array([palette[l] for l in labels])
anchor = TARGET_OURS[labelled]
if HAVE_UMAP:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=SEED, metric="cosine").fit(anchor.numpy())
    reducer_name = "UMAP"
else:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2, random_state=SEED).fit(anchor.numpy())
    reducer_name = "PCA-2D (umap-learn không có)"
panels10 = [("teacher (PCA·R image)", anchor)]
for base in ("pca__procrustes", "learned_t2s__lr1", "learned_s2t__lr1", "simcse_only"):
    if base in SEEDS_OF:
        emb = F.normalize(load_final(SEEDS_OF[base][0])[0][labelled], dim=-1)
        rotation, _ = fit_gauge_alignment(emb, anchor)      # emb @ R ~ anchor
        panels10.append((base, F.normalize(emb @ rotation, dim=-1)))
fig, axes = plt.subplots(1, len(panels10), figsize=(3.4 * len(panels10), 3.6), squeeze=False)
for ax, (title, emb) in zip(axes[0], panels10):
    xy = reducer.transform(emb.numpy())
    ax.scatter(xy[:, 0], xy[:, 1], c=colours, s=4, cmap="tab10")
    ax.set(title=title, xticks=[], yticks=[])
fig.suptitle(f"Figure 10 — {reducer_name} fitted on the PCA·R target, students Procrustes-aligned then transformed (colour = {COLOUR_TASK}; illustration only)")
fig.tight_layout(); savefig(fig, "fig10_2d_embeddings.png"); plt.show()


In [ ]:
# 7. C3 — depth: Figure 4 (CKA tới step 0 theo layer x epoch), Figure 12, Table 5.
depth_bases = [b for b in DEPTH_BASES if b in SEEDS_OF]
depth_rows = []
for base in depth_bases:
    name = SEEDS_OF[base][0]
    layers = load_layers(name)
    epochs = sorted(layers)
    init = layers[0]
    tau, kind = arm_target(name)
    tau_core = (tau if tau is not None else TARGET_OURS)[CORE_INDEX]
    for epoch in epochs:
        for layer in range(init.shape[0]):
            now = layers[epoch][layer]
            depth_rows.append({
                "base": base, "arm": name, "epoch": epoch, "layer": layer,
                "cka_to_init": sa.linear_cka(now, init[layer]),
                "procrustes_to_init": sa.procrustes_distance(now, init[layer]),
                "cos_to_target": float(sa.cosine_to_target(now, tau_core).mean()),
                "twonn_id": sa.twonn_intrinsic_dimension(now, seed=SEED) if epoch in (0, epochs[-1]) else np.nan,
            })
depth = pd.DataFrame(depth_rows)
depth.to_csv(ANALYSIS_DIR / "depth_profiles.csv", index=False)

if depth.empty:
    print("Không có probe_layers nào để vẽ depth.")
else:
    # Figure 4: heatmap CKA (layer x epoch), một panel mỗi arm. Hàng dưới phải ở ~1.0.
    fig, axes = plt.subplots(1, len(depth_bases), figsize=(3.6 * len(depth_bases), 3.8), squeeze=False)
    for ax, base in zip(axes[0], depth_bases):
        sub = depth[(depth["base"] == base) & (depth["epoch"] > 0)].pivot(index="layer", columns="epoch", values="cka_to_init")
        im = ax.imshow(sub.values, vmin=0, vmax=1, cmap="viridis", aspect="auto", origin="lower")
        ax.set(title=base, xlabel="epoch", ylabel="layer (0 = embeddings)", xticks=range(sub.shape[1]), xticklabels=sub.columns)
        for (i, j), v in np.ndenumerate(sub.values):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="w" if v < 0.6 else "k")
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label="linear CKA to step 0")
    fig.suptitle("Figure 4 — how much each layer moved"); savefig(fig, "fig04_depth_heatmap.png"); plt.show()

    # Figure 12: TwoNN ID theo layer (step 0 vs cuối) và cosine tới target theo layer.
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for base in depth_bases:
        sub = depth[depth["base"] == base]
        last = sub["epoch"].max()
        axes[0].plot(sub[sub["epoch"] == last]["layer"], sub[sub["epoch"] == last]["twonn_id"], marker="o", label=f"{base} (end)")
        axes[1].plot(sub[sub["epoch"] == last]["layer"], sub[sub["epoch"] == last]["cos_to_target"], marker="o", label=f"{base} (end)")
    init_sub = depth[(depth["base"] == depth_bases[0]) & (depth["epoch"] == 0)]
    axes[0].plot(init_sub["layer"], init_sub["twonn_id"], color="k", ls="--", marker="x", label="step 0")
    axes[1].plot(init_sub["layer"], init_sub["cos_to_target"], color="k", ls="--", marker="x", label="step 0")
    axes[0].set(title="TwoNN intrinsic dimension by layer", xlabel="layer", ylabel="ID")
    axes[1].set(title="cosine to target by layer", xlabel="layer", ylabel="mean cosine")
    for ax in axes:
        ax.legend(fontsize=8, frameon=False)
    fig.suptitle("Figure 12"); fig.tight_layout(); savefig(fig, "fig12_depth_id_cosine.png"); plt.show()

    # Table 5: full vs frozen-lower + tóm tắt drift (layer thấp vs layer cao) ở epoch cuối.
    table5_rows = []
    for base in depth_bases:
        sub = depth[(depth["base"] == base) & (depth["epoch"] == depth[depth["base"] == base]["epoch"].max())]
        n_layers = int(sub["layer"].max())
        lower = sub[sub["layer"].between(1, n_layers - 2)]["cka_to_init"].mean()
        upper = sub[sub["layer"] >= n_layers - 1]["cka_to_init"].mean()
        scores = [score_of(n) for n in SEEDS_OF[base] if score_of(n) is not None]
        table5_rows.append({"base": base, "avg_all_mean": np.mean(scores) if scores else np.nan, "avg_all_std": np.std(scores) if len(scores) > 1 else np.nan,
                            "cka_lower_layers": lower, "cka_top_two": upper})
    table5 = pd.DataFrame(table5_rows)
    table5.to_csv(ANALYSIS_DIR / "table5_depth.csv", index=False)
    print("TABLE 5 — depth (CKA to step 0: lower stack vs top two blocks)")
    display(table5.style.format(precision=4, na_rep="-"))
    if "freeze_lower" not in SEEDS_OF:
        print("[note] freeze_lower chưa có (thiếu code): Table 5 chỉ có cột drift, chưa có so sánh điểm.")


In [ ]:
# 8. G — gauge: Table 6 (null band, PR) và Figure 5 (interpolation nếu có).
def theta_of(base):
    """Vị trí của arm trên trục nội suy: 0 = Procrustes R, 1 = Haar Q; None = không nằm trên trục."""
    if base == "pca__procrustes":
        return 0.0
    if base.startswith("pca__random"):
        return 1.0
    if base.startswith("gauge_theta"):
        return float(base.replace("gauge_theta", ""))
    return None


gauge_rows = []
for base in BASES:
    if not (base.startswith("pca__") or base.startswith("gauge_")) or base == "pca__mse":
        continue
    for name in SEEDS_OF[base]:
        summary = ARMS[name]["summary"] or {}
        stats = summary.get("gauge_stats") or {}
        by_epoch = summary.get("train_by_epoch") or []
        theta = theta_of(base)
        gauge_rows.append({
            "base": base, "arm": name, "theta": theta,
            "avg_all": score_of(name), "loss_end_epoch1": (by_epoch[0] or {}).get("loss_end") if by_epoch else None,
            "final_loss_end": final_train(name, "loss_end"),
            "participation_ratio": stats.get("participation_ratio"), "top_singular_share": stats.get("top_singular_share"),
            "cos_init_before": stats.get("cos_before"), "cos_init_after": stats.get("cos_after"), "cos_procrustes_would": stats.get("cos_procrustes"),
        })
gauge = pd.DataFrame(gauge_rows)
if gauge.empty:
    raise RuntimeError("Không có arm pca__* nào có probe dump — chạy C1 trước khi đọc G.")
gauge.to_csv(ANALYSIS_DIR / "table6_gauge_by_arm.csv", index=False)
gauge["family"] = gauge["base"].str.replace(r"__d\d+$", "", regex=True)
table6 = gauge.groupby("family", sort=False)[["avg_all", "loss_end_epoch1", "final_loss_end", "participation_ratio", "cos_init_after"]].agg(["mean", "std", "count"])
table6.columns = [f"{c}_{s}" for c, s in table6.columns]
table6.to_csv(ANALYSIS_DIR / "table6_gauge.csv")
print("TABLE 6 — gauge (pca__random: mean ± std over draws = null band)")
display(table6.style.format(precision=4, na_rep="-"))

pr = gauge["participation_ratio"].dropna()
if not pr.empty:
    print(f"participation ratio của cặp này: {pr.iloc[0]:.2f} / {D_S} — "
          + ("gần hạng 1: gauge được DỰ ĐOÁN gần null trên cặp này" if pr.iloc[0] < 3 else "nhiều chiều thật để khớp"))

# Figure 5: x = theta (0 Procrustes ... 1 random), y = avg_all và loss_end epoch 1; band = random draws.
with_theta = gauge[gauge["theta"].notna()]
if not with_theta.empty:
    fig, ax1 = plt.subplots(figsize=(7, 4.5))
    ax2 = ax1.twinx()
    agg = with_theta.groupby("theta")[["avg_all", "loss_end_epoch1"]].agg(["mean", "std"])
    ax1.errorbar(agg.index, agg[("avg_all", "mean")], yerr=agg[("avg_all", "std")].fillna(0), marker="o", capsize=3, color="C0", label="avg_all")
    ax2.errorbar(agg.index, agg[("loss_end_epoch1", "mean")], yerr=agg[("loss_end_epoch1", "std")].fillna(0), marker="s", capsize=3, color="C3", ls="--", label="endpoint loss, epoch 1")
    rand = gauge[gauge["base"].str.startswith("pca__random")]["avg_all"].dropna()
    if len(rand) > 1:
        ax1.axhspan(rand.mean() - rand.std(), rand.mean() + rand.std(), color="gray", alpha=0.2, label="random-rotation null band")
    none_score = gauge[gauge["base"] == "pca__none"]["avg_all"].mean()
    if not np.isnan(none_score):
        ax1.axhline(none_score, color="C2", ls=":", label="pca__none (no rotation)")
    ax1.set(title=f"Figure 5 — gauge interpolation (PR = {pr.iloc[0]:.2f})" if not pr.empty else "Figure 5", xlabel="theta: 0 = Procrustes R, 1 = Haar Q", ylabel="avg_all (x100)")
    ax2.set_ylabel("endpoint loss after epoch 1")
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=8, frameon=False, loc="best")
    fig.tight_layout(); savefig(fig, "fig05_gauge_interpolation.png"); plt.show()
if not any(b.startswith("gauge_theta") for b in BASES):
    print("[note] Chưa có gauge_theta{0.25,0.5,0.75} / gauge_rank_one (thiếu code): Figure 5 chỉ có hai đầu mút 0 và 1.")


In [ ]:
# 9. M — Figure 13 (truncation), corpus sweep, Table 7 (matched-TALAS).
trunc_rows = []
for base in ("pca__procrustes", "learned_s2t__lr1", "learned_t2s__lr1", "simcse_only", "pca__mse"):
    if base not in SEEDS_OF:
        continue
    for name in SEEDS_OF[base]:
        final, _ = load_final(name)
        for row in sa.truncation_curve(final, PAIRS, list(TRUNCATION_DIMS)):
            trunc_rows.append({"base": base, "arm": name, "kind": PAIRS[row["task"]]["kind"], **row})
trunc = pd.DataFrame(trunc_rows)
if not trunc.empty:
    trunc.to_csv(ANALYSIS_DIR / "fig13_truncation.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, kind, label in ((axes[0], "sts", "STS Spearman (mean of 3)"), (axes[1], "pair", "pair AP (mean of 3)")):
        for base, sub in trunc[trunc["kind"] == kind].groupby("base", sort=False):
            agg = sub.groupby("dim")["score"].mean()
            ax.plot(agg.index, 100 * agg.values, marker="o", label=base)
        ax.set(title=label, xlabel="retained leading dimensions", ylabel="score (x100)")
        ax.legend(fontsize=8, frameon=False)
    fig.suptitle("Figure 13 — MRL-style truncation of the distilled student"); fig.tight_layout(); savefig(fig, "fig13_truncation.png"); plt.show()

# Corpus-size sweep: ours vs baseline mạnh nhất theo mức corpus.
sweep_rows = [{"base": ARMS[n]["base"], "dataset": ARMS[n]["dataset"], "method": ARMS[n]["method"], "seed": ARMS[n]["seed"], "avg_all": score_of(n)}
              for n in DONE if ARMS[n]["base"].startswith(("ours__", "talas__", "pca__procrustes"))]
sweep = pd.DataFrame(sweep_rows)
if not sweep.empty:
    sweep["method"] = sweep["base"].map(lambda b: "ours" if b.startswith(("ours__", "pca__procrustes")) else b.split("__")[0])
    pivot = sweep.pivot_table(index="dataset", columns="method", values="avg_all", aggfunc="mean")
    pivot.to_csv(ANALYSIS_DIR / "corpus_sweep.csv")
    print("CORPUS SWEEP (avg_all x100)")
    display(pivot.style.format(precision=2, na_rep="-"))

# Table 7: matched-HP protocol vs TALAS.
table7 = pd.DataFrame([
    {"row": base, "avg_all_mean": np.mean([score_of(n) for n in SEEDS_OF[base] if score_of(n) is not None]) if SEEDS_OF.get(base) else np.nan,
     "n_seeds": len(SEEDS_OF.get(base, [])), "dataset": ARMS[SEEDS_OF[base][0]]["dataset"] if SEEDS_OF.get(base) else "-"}
    for base in ("pca__procrustes", "talas__matched", "talas__adamw", "ours__talas15k")
])
table7.to_csv(ANALYSIS_DIR / "table7_matched_talas.csv", index=False)
print("TABLE 7 — matched-HP protocol vs TALAS")
display(table7.style.format(precision=2, na_rep="-"))
if "talas__adamw" not in SEEDS_OF:
    print("[note] talas__adamw chưa có (thiếu code --talas_optimizer): hàng đó trống.")


In [ ]:
# 10. Danh sách output.
for path in sorted(ANALYSIS_DIR.iterdir()):
    print(f"{path.stat().st_size / 1024:8.1f} KB  {path.name}")
print(f"\nMọi bảng/figure nằm ở: {ANALYSIS_DIR}")
